# RockWatch - Phase 4: Advanced Machine Learning Models
**Project:** Intelligent Landslide Susceptibility Prediction System for Himachal Pradesh  
**Objective:** Train, tune via 5-Fold Spatial Block Cross-Validation, and benchmark advanced ensemble models (`RandomForestClassifier`, `GradientBoostingClassifier`, `SVC`).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# Set project root
PROJECT_ROOT = Path('.').resolve().parents[0] if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_processed_data
from src.models.baseline import ALL_FEATURES, LR_FEATURES

## 1. Load Processed Datasets

In [ ]:
f_train = load_processed_data('train', features_only=True, processed_dir=PROJECT_ROOT / 'data' / 'processed')
f_val = load_processed_data('val', features_only=True, processed_dir=PROJECT_ROOT / 'data' / 'processed')
f_test = load_processed_data('test', features_only=True, processed_dir=PROJECT_ROOT / 'data' / 'processed')

y_train = f_train['class'].values
y_val = f_val['class'].values
y_test = f_test['class'].values

print(f'Train: {len(f_train):,} | Val: {len(f_val):,} | Test: {len(f_test):,}')

## 2. Overall Model Leaderboard (All 6 Models)

In [ ]:
leaderboard = pd.read_csv(PROJECT_ROOT / 'models' / 'evaluation' / 'all_models_benchmark.csv')
leaderboard[['Model', 'Spatial_CV_AUC', 'Val_AUC', 'Test_AUC', 'Test_PR_AUC', 'Test_F1_opt', 'Test_MCC_opt', 'Test_Accuracy_opt']]

## 3. Feature Importance Analysis (Random Forest vs Gradient Boosting)

In [ ]:
rf = joblib.load(PROJECT_ROOT / 'models' / 'trained' / 'random_forest_v1.joblib')
gb = joblib.load(PROJECT_ROOT / 'models' / 'trained' / 'gradient_boosting_v1.joblib')

imp_df = pd.DataFrame({
    'Feature': ALL_FEATURES,
    'Random_Forest': rf.feature_importances_,
    'Gradient_Boosting': gb.feature_importances_,
}).sort_values('Gradient_Boosting', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
y_pos = np.arange(len(imp_df))
ax.barh(y_pos - 0.2, imp_df['Random_Forest'], height=0.4, label='Random Forest', color='#2b83ba')
ax.barh(y_pos + 0.2, imp_df['Gradient_Boosting'], height=0.4, label='Gradient Boosting', color='#d7191c')
ax.set_yticks(y_pos)
ax.set_yticklabels(imp_df['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Relative Feature Importance')
ax.set_title('Feature Importance Comparison: Random Forest vs Gradient Boosting')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Comparative ROC Curves on Held-out Spatial Test Split (Chamba)

In [ ]:
models = {
    'Logistic Regression': (joblib.load(PROJECT_ROOT / 'models' / 'trained' / 'logistic_regression_v1.joblib'), LR_FEATURES),
    'Decision Tree (d=5)': (joblib.load(PROJECT_ROOT / 'models' / 'trained' / 'decision_tree_v1.joblib'), ALL_FEATURES),
    'SVM (RBF Kernel)': (joblib.load(PROJECT_ROOT / 'models' / 'trained' / 'svm_rbf_v1.joblib'), ALL_FEATURES),
    'Random Forest': (rf, ALL_FEATURES),
    'Gradient Boosting': (gb, ALL_FEATURES),
}

fig, ax = plt.subplots(figsize=(9, 6), dpi=150)
for name, (clf, feats) in models.items():
    p_test = clf.predict_proba(f_test[feats].values)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, p_test)
    ax.plot(fpr, tpr, label=f'{name} (Test AUC = {auc(fpr, tpr):.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_title('RockWatch Model Benchmark - ROC Curves on Held-out Spatial Test Set')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.grid(True, linestyle=':', alpha=0.6)
plt.show()